In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print(f"Total lesson pages: {len(documents)}")

Total lesson pages: 72


In [ ]:
import minsearch

index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(query=query, num_results=5)

first_result_filename = search_results[0]['filename']
print(f"First result filename: {first_result_filename}")

First result filename: 01-agentic-rag/lessons/14-agentic-loop.md


In [ ]:
from google import genai

client = genai.Client()


PROMPT_TEMPLATE = """
You are a course teaching assistant. Answer the user QUERY using only the provided CONTEXT.

QUERY: {query}

CONTEXT:
{context}
""".strip()

def build_context(search_results):
    context_parts = []
    for doc in search_results:
        context_parts.append(f"Filename: {doc['filename']}\nContent: {doc['content']}\n")
    return "\n\n".join(context_parts)

def rag_with_gemini(query):
    search_results = index.search(query=query, num_results=5)
    
    context = build_context(search_results)
    
    prompt = PROMPT_TEMPLATE.format(query=query, context=context)
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    
    input_tokens = response.usage_metadata.prompt_token_count
    
    return response.text, input_tokens

query = "How does the agentic loop keep calling the model until it stops?"
answer, tokens_used = rag_with_gemini(query)

print(f"Prompt Tokens Sent: {tokens_used}")


Prompt Tokens Sent: 7924


In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks generated: {len(chunks)}")

Total chunks generated: 295


In [ ]:
chunk_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks) 

def rag_with_chunks(query):
    search_results = chunk_index.search(query=query, num_results=5)
    
    context = build_context(search_results) 
    prompt = PROMPT_TEMPLATE.format(query=query, context=context)
    
    client = genai.Client()
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )
    
    return response.usage_metadata.prompt_token_count

query = "How does the agentic loop keep calling the model until it stops?"
chunked_tokens = rag_with_chunks(query)

print(f"Tokens with chunking: {chunked_tokens}") # 3x fewer

Tokens with chunking: 2577


In [8]:

from google.genai import types

def course_lessons_search(query: str) -> str:
    """
    Search the course lesson knowledge base chunks for relevant information using keywords.
    
    Args:
        query: The search keywords or phrase.
    """
    global call_counter
    call_counter += 1  
    
    results = chunk_index.search(query=query, num_results=5)
    
    return build_context(results)

call_counter = 0

system_instruction = (
    "You're a course teaching assistant. Answer the student's question using the search tool. "
    "Make multiple searches with different keywords before answering."
)

user_message = "How does the agentic loop work, and how is it different from plain RAG?"

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        tools=[course_lessons_search], 
    )
)

response = chat.send_message(user_message)

print(f"Total search tool calls executed: {call_counter}")


Total search tool calls executed: 2
